In [1]:
import numpy as np
from matplotlib import pyplot as plt
from electronic import *
from analytical import *
from integrator import *

In [ ]:
# PC-REV LD result
v = 0.002
c = 0.0033
s = 0.66
tau = 4 * np.abs(c)/np.abs(s * v)
P12 = 1 - np.exp(-2 * np.pi * c**2 / abs(s * v))

LZ_H = LandauZener(s=s, vc=c)
LZ_H.mass = 1822.89
dt = 10.0 # here to vary the length of the timestep
n_int = 5.0/(v * dt)
t0 = dt * (n_int + 3/10)
t = np.arange(0, 8269, dt)
traj = np.array([Landau_traj(v=v, t=tn, t0=t0) for tn in t])

q_start = traj[0]
C_curr = LZ_H.eigvecs(q=q_start)
coeff_curr = np.array([1.0,0.0], dtype=complex)
q_curr = q_start
v_curr = v
hop_exists = False


for step in range(len(traj)-1):
    Sz_curr = sz_from_coeff(coeff_curr)
    active_state = get_active_state(Sz_curr)
    F_curr = LZ_H.F(a=active_state, q = q_curr)
    v_next_half = verlet_v(dt, v_curr, F_curr)
    q_next = q_curr + dt * v_next_half
    F_next = LZ_H.F(a=active_state, q = q_next)
    v_next = verlet_v(dt, v_next_half, F_next)
    C_next, coeff_next = LD_step(LZ_H, q_curr, q_next, dt, C_curr, coeff_curr)
    Sz_next = sz_from_coeff(coeff_next)
    if Sz_curr * Sz_next < 0:
        tau_M, q_M, v_M, C_M, coeff_M = hop_search_direct(LZ_H, dt, q_curr, q_next, C_curr, coeff_curr, Sz_curr, Sz_next, tol_sz=1e-12, tol_tau=1e-10)
        hop_exists, v_M, Sz
        C_curr = C_M
        coeff_curr = coeff_M
        C_next, coeff_next = LD_step(LZ_H, q_M, q_next, dt-tau_M, C_curr, coeff_curr)
    C_curr = C_next
    coeff_curr = coeff_next

C_final_fwd = C_curr

print(np.abs(coeff_next[0])**2)
print(coeff_next)